# Run the S2 MSI raw generator (ESA-only NUC)

End-to-end run of the reverse chain on the Studio VM using the **ESA EOPF `R2EQOG` ADF** as the NUC
source (branch `feat/esa-eopf-r2eqog-nuc`). Steps:

1. Update the repo checkout and (re)install the package.
2. Environment + discover an available L1A input.
3. Run the pipeline (`preflight` → `radiometric-vv` applies the ESA NUC + temporal-validity check).
4. Round-trip validation on a ESA SAFE L1B granule.
5. Inspect the reports.

## 1) Update the repo and install

Detect the repo root, pull the branch, and install in editable mode so the new `gipp`/ADF code is used.

In [1]:
import os
from pathlib import Path

current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")
repo_name = "s2-msi-raw-generator"
REPO_ROOT = Path(current_dir) / repo_name
print(f"Repository root: {REPO_ROOT}")




Current working directory: /home/jovyan
Repository root: /home/jovyan/s2-msi-raw-generator


In [2]:
# go to repo root, fetch + checkout the feature branch, pull, install (commands)
!cd {REPO_ROOT} && git fetch origin && git checkout feat/esa-eopf-r2eqog-nuc && git pull --ff-only
!cd {REPO_ROOT} && pip install -e . --quiet && echo "installed"
!cd {REPO_ROOT} && git log --oneline -5

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 358.28 KiB | 12.35 MiB/s, done.
From https://gitlab.eopf.copernicus.eu/e2es/s2-msi-raw-generator
   a864934..3db34f8  feat/esa-eopf-r2eqog-nuc -> origin/feat/esa-eopf-r2eqog-nuc
D	s2_msi_raw_generator/data/psf/PROVENANCE.md
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B01.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B02.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B03.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B04.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B05.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B06.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B07.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B08.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B09.csv
D	s2_msi_raw_generator/data/psf/S2A/S2A_PSF_B11.csv
D	s2_msi_raw_generator/dat

## 2) Environment + input discovery

Point the pipeline at the ESA EOPF `R2EQOG` ADF, then discover an available L1A product in the
data-store (and the bands it actually contains) so we can run `preflight → radiometric-vv` directly —
this is the phase that applies the ESA NUC and runs the temporal-validity check, so we skip the
synthetic `import-l0` step (which needs a public-L0 input that may be absent).

In [3]:
import os

DATA_STORE = os.path.expanduser("~/data-store")
os.environ["S2_DATA_STORE"] = DATA_STORE
# ESA equalization/NUC ADF (downloaded to the data-store); comment out to fall back to the XML GIPP
os.environ["S2_E2ES_EQOG_ADF"] = (
    f"{DATA_STORE}/esa-adf/S2A_ADF_REQOG_20240417T000000_21000101T000000_20240417T000000.json"
)
os.environ["S2_E2ES_LINES"] = "2048"          # along-track window for a quick run (0 = full frame)

print("NUC source :", os.environ["S2_E2ES_EQOG_ADF"], "| exists:", os.path.exists(os.environ["S2_E2ES_EQOG_ADF"]))
gipp_dir = os.environ.get("S2_E2ES_GIPP_DIR", f"{DATA_STORE}/inputs/s2-sensor/GIPP")
print("GIPP dir   :", gipp_dir, "| exists:", os.path.isdir(gipp_dir))

NUC source : /home/jovyan/data-store/esa-adf/S2A_ADF_REQOG_20240417T000000_21000101T000000_20240417T000000.json | exists: True
GIPP dir   : /home/jovyan/data-store/inputs/s2-sensor/GIPP | exists: True


In [4]:
import glob
import re

from s2_msi_raw_generator import io as gio, sensor

cands = sorted(set(
    glob.glob(f"{DATA_STORE}/inputs/**/*L1A*.zarr", recursive=True)
    + glob.glob(f"{DATA_STORE}/inputs/PDI_MSI_S2_L1A.zarr")
))

def bands_of(path):
    # bands under measurements/DD<det>/<band>/l1a_raw_image (the raw-DN L1A layout read_l1a_raw expects)
    try:
        gio.read_l1a_raw(path, 1, "B01", lines=slice(0, 4))   # probe: is this a readable raw-DN L1A?
    except Exception:
        return []
    try:
        g = gio._open(path)
        meas = g["measurements"] if "measurements" in g else g
        found = set()
        for dnm, det in meas.groups():
            if not re.fullmatch(r"DD\d{2}", dnm):
                continue
            for bnm, _ in det.groups():
                if re.fullmatch(r"B(\d{2}|8A)", bnm):
                    found.add(bnm)
        return sorted(found & set(sensor.BANDS))
    except Exception:
        return []

best = None
for p in cands:
    b = bands_of(p)
    print(f"  {os.path.relpath(p, DATA_STORE)}: raw-L1A readable, {len(b)} bands" if b
          else f"  {os.path.relpath(p, DATA_STORE)}: not a raw-DN L1A (skip)")
    if b and (best is None or len(b) > len(best[1])):
        best = (p, b)

if best:
    os.environ["S2_E2ES_L1A"] = best[0]
    os.environ["S2_E2ES_BANDS"] = ",".join(best[1])
    os.environ["S2_E2ES_PHASES"] = "preflight,radiometric-vv"
    print(f"\n→ pipeline L1A : {best[0]}  ({len(best[1])} bands)")
    print(f"→ phases       : preflight,radiometric-vv  (raw->forward->reverse round-trip V&V)")
else:
    print("\n→ no raw-DN L1A found — run step 4 (validate_esa_nuc.py) instead")


  inputs/PDI_MSI_S2_L1A.zarr: raw-L1A readable, 13 bands
  inputs/public-data/level-1/S02MSIL1A_20240403T000000_0001_A123_T000.zarr: not a raw-DN L1A (skip)
  inputs/s2-sensor/PDI_MSI_S2_L1A/PDI_MSI_S2_L1A.zarr: raw-L1A readable, 13 bands
  inputs/s2-sensor/RADIO_AB_input/PDI_MSI_S2_L1A.zarr: not a raw-DN L1A (skip)

→ pipeline L1A : /home/jovyan/data-store/inputs/PDI_MSI_S2_L1A.zarr  (13 bands)
→ phases       : preflight,radiometric-vv  (raw->forward->reverse round-trip V&V)


## 3) Run the pipeline (ESA NUC path)

Runs `preflight → radiometric-vv` on the discovered L1A. `radiometric-vv` applies the ESA `R2EQOG`
NUC and emits the temporal-validity check (a WARNING when the ADF epoch is stale vs the acquisition).

> Full synthetic run instead? Once a public L0 is present, set
> `S2_E2ES_PHASES=import-l0,preflight,package,ground-decode,l0-decode,validate,radiometric-vv,inventory,report`
> (or just remove `S2_E2ES_PHASES`) and run `nominal`.

In [5]:
!cd {REPO_ROOT} && python scripts/run_pipeline.py nominal

[preflight] B01: {'shape': [2048, 1296], 'min': 48, 'max': 1155, 'n_gt_4095': 0, 'n_sentinel_32768': 0, 'trailing_zero_lines': 0, 'entropy_bits_px': 5.087}
[preflight] B02: {'shape': [2048, 2592], 'min': 48, 'max': 1099, 'n_gt_4095': 0, 'n_sentinel_32768': 0, 'trailing_zero_lines': 0, 'entropy_bits_px': 4.891}
[preflight] B03: {'shape': [2048, 2592], 'min': 48, 'max': 1076, 'n_gt_4095': 0, 'n_sentinel_32768': 0, 'trailing_zero_lines': 0, 'entropy_bits_px': 5.129}
[preflight] B04: {'shape': [2048, 2592], 'min': 48, 'max': 1038, 'n_gt_4095': 0, 'n_sentinel_32768': 0, 'trailing_zero_lines': 0, 'entropy_bits_px': 5.21}
[preflight] B05: {'shape': [2048, 1296], 'min': 48, 'max': 1099, 'n_gt_4095': 0, 'n_sentinel_32768': 0, 'trailing_zero_lines': 0, 'entropy_bits_px': 4.57}
[preflight] B06: {'shape': [2048, 1296], 'min': 47, 'max': 961, 'n_gt_4095': 0, 'n_sentinel_32768': 0, 'trailing_zero_lines': 0, 'entropy_bits_px': 4.42}
[preflight] B07: {'shape': [2048, 1296], 'min': 47, 'max': 835, 'n_g

## 4) Round-trip validation on a ESA SAFE L1B granule

Re-impress the NUC with the ESA `R2EQOG` on a genuine ESA L1B jp2 and forward-correct it back —
reports the round-trip RMSE, the column-FPN increase (proof the ESA per-pixel dark/PRNU was
impressed), and the ADF temporal validity vs the acquisition epoch.

In [6]:
!cd {REPO_ROOT} && python scripts/validate_esa_nuc.py --band B03 --detector 1 --json /tmp/esa_nuc_val.json

L1B : /home/jovyan/validation-data/data-store/dpr-common-validation/turkey/S20180820T085005_L1B.tar
ADF : /home/jovyan/data-store/esa-adf/S2A_ADF_REQOG_20240417T000000_21000101T000000_20240417T000000.json
band B03  detector 1

granule: S20180820T085005_L1B/S2A_OPER_MSI_L1B_GR_2BPS_20250611T101114_S20180820T084951_D01_N05.11.tar
L1B image: shape=(2048, 2552) mean=3309.9 std=992.3
ESA R2EQOG: model=CUBIC act_px=2592 dark_mean=440.8 rel_gain[:3]=[1. 1. 1.]

=== ESA-NUC round-trip (S2 L1B pixels) ===
round-trip RMSE     : 0.0000 DN  (0.0000 % of mean)
column FPN  L1B/raw : 0.0072 -> 0.0053  (no FPN increase (check ADF/band))
synthetic raw       : mean=3389.7 std=704.6

=== temporal validity ===
WARNING — ADF REQOG applicability 2024-04-17 vs acquisition 2018-08-20: gap 5.66 yr — OUTSIDE declared validity window — TEMPORAL MISMATCH (>1 yr)

wrote /tmp/esa_nuc_val.json


## 5) Inspect the reports

Pretty-print the radiometric-VV report (per-band RMSE + the `_adf_temporal_validity` block) and the
validation json.

In [7]:
import json
from pathlib import Path

store = Path(os.environ["S2_DATA_STORE"])
rvv = store / "report/radiometric_vv.json"
if rvv.exists():
    data = json.loads(rvv.read_text())
    tv = data.pop("_adf_temporal_validity", None)
    print("=== radiometric_vv.json (per-band) ===")
    for band, m in data.items():
        print(f"  {band}: {m}")
    print("\n=== ADF temporal validity ===")
    print(json.dumps(tv, indent=2))
else:
    print("radiometric_vv.json not found — did step 3 reach the radiometric-vv phase?")

val = Path("/tmp/esa_nuc_val.json")
if val.exists():
    print("\n=== validate_esa_nuc.json ===")
    print(json.dumps(json.loads(val.read_text()), indent=2))

=== radiometric_vv.json (per-band) ===
  B01: {'rmse': 1.157201682582371e-16, 'fpn_raw': 0.10178583249010004, 'fpn_corrected': 0.19922751223175136}
  B02: {'rmse': 3.581487796809445e-15, 'fpn_raw': 0.06018326643892351, 'fpn_corrected': 0.331236433239204}
  B03: {'rmse': 5.879227517228008e-15, 'fpn_raw': 0.04834200653354531, 'fpn_corrected': 0.2970797633692287}
  B04: {'rmse': 8.856614317138858e-15, 'fpn_raw': 0.03736683914390092, 'fpn_corrected': 0.300492045919746}
  B05: {'rmse': 1.0470422771922719e-14, 'fpn_raw': 0.05031316855203543, 'fpn_corrected': 0.32073531764445606}
  B06: {'rmse': 1.0608424841172242e-14, 'fpn_raw': 0.04324102233485995, 'fpn_corrected': 0.2791789078681275}
  B07: {'rmse': 1.2062039384214223e-14, 'fpn_raw': 0.0378102208510226, 'fpn_corrected': 0.21071248828508016}
  B08: {'rmse': 1.058122153421506e-14, 'fpn_raw': 0.02578722097456144, 'fpn_corrected': 0.2459516170699713}
  B09: {'rmse': 1.4403421421252992e-14, 'fpn_raw': 0.016293861783228517, 'fpn_corrected': 0.0}

## 6) Before / after figures (visual check)

Plot the ESA-NUC transformation for eyeball inspection, with statistics:

* **A) Pipeline** — raw L1A → ESA-equalized L1B (`forward_correct`).
* **B) Real turkey L1B** — S2 L1B → synthetic L0 raw (`reverse_impress`, the generator output).

Each row shows the two images plus the **across-track column-mean profile** (the fixed-pattern-noise
signature) with the column-FPN metric, so the PRNU/dark effect of the ESA ADF is visible.

In [8]:
# 6) before/after figure + stats  (ONE S2 L1B input: turkey L1B -> synthetic raw Synthetic L0)
import io as _io
import os
import tarfile

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    import subprocess, sys as _s
    subprocess.run([_s.executable, "-m", "pip", "install", "-q", "matplotlib"])
    import matplotlib.pyplot as plt

from s2_msi_raw_generator import forward_radiometric_atbd as fwd, gipp

# reverse_impress needs an EQUALIZED DN image. The turkey L1B (jp2, ~DN) is exactly that;
# the pipeline's PDI is already raw-DN and the EOPF radiance products are in physical units,
# so neither is the right domain for a generation figure — we use the S2 L1B we downloaded.
ADF = os.environ["S2_E2ES_EQOG_ADF"]
L1B_TAR = ("/home/jovyan/validation-data/data-store/dpr-common-validation/turkey/"
           "S20180820T085005_L1B.tar")
BAND, DET, LINES = "B03", 1, 1024
eq = gipp.read_r2eqog_eopf(ADF, BAND).detectors[DET]


def _read_l1b_jp2(tar, det, band):
    from PIL import Image
    with tarfile.open(tar) as outer:
        gran = [m for m in outer.getmembers() if f"_D{det:02d}_" in m.name and m.name.endswith(".tar")]
        with tarfile.open(fileobj=_io.BytesIO(outer.extractfile(gran[0]).read())) as inner:
            jp2 = [m for m in inner.getmembers() if m.name.endswith(f"_{band}.jp2")]
            return np.asarray(Image.open(_io.BytesIO(inner.extractfile(jp2[0]).read())), dtype=float)


def _stats(a):
    v = a[a > 0]; v = v if v.size else a
    return f"mean={v.mean():.1f}  std={v.std():.1f}  min={v.min():.0f}  max={v.max():.0f}  FPN={fwd.column_fpn(a):.4f}"


if not os.path.exists(L1B_TAR):
    print("turkey L1B not found:", L1B_TAR)
else:
    y = _read_l1b_jp2(L1B_TAR, DET, BAND)[:LINES]     # INPUT: S2 equalized L1B (DN)
    x = fwd.reverse_impress(y, eq)                    # OUTPUT: synthetic raw L0
    print("INPUT   S2 L1B :", _stats(y))
    print("OUTPUT  raw L0   :", _stats(x))

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
    for k, (img, t) in enumerate([(y, "INPUT  S2 L1B (equalized)"),
                                  (x, "OUTPUT  synthetic raw L0")]):
        m = img[img > 0]
        lo, hi = np.percentile(m, (2, 98)) if m.size else (img.min(), img.max())
        im = ax[k].imshow(np.clip(img, lo, max(hi, lo + 1e-9)), cmap="gray", aspect="auto",
                          interpolation="nearest")
        ax[k].set_title(f"{t}\n{_stats(img)}", fontsize=8)
        ax[k].set_xlabel("across-track px"); ax[k].set_ylabel("line")
        plt.colorbar(im, ax=ax[k], fraction=0.046, pad=0.02)
    ax[2].plot(y.mean(0), lw=0.7, label=f"input L1B  (FPN {fwd.column_fpn(y):.4f})")
    ax[2].plot(x.mean(0), lw=0.7, label=f"output raw L0  (FPN {fwd.column_fpn(x):.4f})")
    ax[2].set_title("across-track column mean (before vs after)", fontsize=9)
    ax[2].set_xlabel("column"); ax[2].set_ylabel("mean DN"); ax[2].legend(fontsize=8)
    fig.suptitle(f"turkey L1B  {BAND} d{DET:02d}  —  reverse_impress -> synthetic raw L0 "
                 f"(ESA R2EQOG {os.path.basename(ADF)[:20]}...)", fontsize=10)
    fig.tight_layout(); plt.show()


INPUT   S2 L1B : mean=3039.0  std=922.5  min=1824  max=10936  FPN=0.0093
OUTPUT  raw L0   : mean=3191.2  std=673.0  min=2200  max=7245  FPN=0.0067


## 7) Reverse chain, step by step

Run the reverse radiometric chain on the turkey L1B and show the image **after each step**
(`reverse.py`, ATBD §5). The input is the equalized L1B DN (the post-S1 state), then:
**S6** PSF re-blur → **S7** impress PRNU → **S13** add noise → **S11** reapply dark →
**S12** onboard equalization → **S14** quantize (12-bit) = **synthetic L0 raw**. All coefficients
come from the ESA `R2EQOG` ADF.

In [9]:
# 7) reverse chain step-by-step on the turkey L1B
import io as _io
import os
import tarfile

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    import subprocess, sys as _s
    subprocess.run([_s.executable, "-m", "pip", "install", "-q", "matplotlib"])
    import matplotlib.pyplot as plt

from s2_msi_raw_generator import adf as adfmod, gipp, reverse, sensor

ADF = os.environ["S2_E2ES_EQOG_ADF"]
L1B_TAR = ("/home/jovyan/validation-data/data-store/dpr-common-validation/turkey/"
           "S20180820T085005_L1B.tar")
BAND, DET, LINES = "B03", 1, 1024
rng = np.random.default_rng(0)


def _read_l1b_jp2(tar, det, band):
    from PIL import Image
    with tarfile.open(tar) as outer:
        gran = [m for m in outer.getmembers() if f"_D{det:02d}_" in m.name and m.name.endswith(".tar")]
        with tarfile.open(fileobj=_io.BytesIO(outer.extractfile(gran[0]).read())) as inner:
            jp2 = [m for m in inner.getmembers() if m.name.endswith(f"_{band}.jp2")]
            return np.asarray(Image.open(_io.BytesIO(inner.extractfile(jp2[0]).read())), dtype=float)


y = _read_l1b_jp2(L1B_TAR, DET, BAND)[:LINES]

# BandADF from the ESA R2EQOG (prnu/dark/eq/psf/noise), aligned to the L1B active width
gs = gipp.GippSet(gipp_dir="")
gs.equalization[BAND] = gipp.read_r2eqog_eopf(ADF, BAND)
a = adfmod.BandADF.from_gipp(sensor.band(BAND), DET, gs, active_width=y.shape[1])

# apply each reverse step, keeping the intermediate image
steps = [("input: L1B equalized DN", y)]
z = reverse.s6_psf_reblur(y, a.psf);                          steps.append(("S6  PSF re-blur", z))
z = reverse.s7_impress_relative_response(z, a.prnu_gain);     steps.append(("S7  impress PRNU", z))
z = reverse.s13_add_noise(z, a.noise_a, a.noise_b, rng);      steps.append(("S13 add noise", z))
z = reverse.s11_reapply_dark(z, a.dark_dn);                   steps.append(("S11 reapply dark", z))
z = reverse.s12_reapply_onboard_eq(z, a.eq_gain, a.eq_offset); steps.append(("S12 onboard eq", z))
z = reverse.s14_quantize(z);                                  steps.append(("S14 quantize = L0 raw", z))
L0_synth = np.asarray(z, dtype=float)

print("step                        mean      std     min     max")
for name, img in steps:
    v = np.asarray(img, float); m = v[v > 0] if (v > 0).any() else v
    print(f"  {name:26s} {m.mean():8.1f} {m.std():7.1f} {m.min():7.0f} {m.max():7.0f}")

n = len(steps)
cols = 4; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.4 * rows))
for ax, (name, img) in zip(axes.ravel(), steps):
    v = np.asarray(img, float); m = v[v > 0]
    lo, hi = np.percentile(m, (2, 98)) if m.size else (v.min(), v.max())
    im = ax.imshow(np.clip(v, lo, max(hi, lo + 1e-9)), cmap="gray", aspect="auto",
                   interpolation="nearest")
    ax.set_title(f"{name}\nmean={v[v>0].mean():.0f} std={v[v>0].std():.0f}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.suptitle(f"Reverse chain step-by-step — turkey {BAND} d{DET:02d} (ESA R2EQOG)", fontsize=11)
fig.tight_layout(); plt.show()


step                        mean      std     min     max
  input: L1B equalized DN      3039.0   922.5    1824   10936
  S6  PSF re-blur              3039.0   922.5    1824   10936
  S7  impress PRNU             3061.3   932.0    1828   11072
  S13 add noise                3061.3   932.1    1832   11051
  S11 reapply dark             3502.2   932.2    2271   11491
  S12 onboard eq               3502.2   932.2    2271   11491
  S14 quantize = L0 raw        3324.6   607.6    2271    4095


## 8) Fetch the EOPF zarr L0 (same acquisition)

We cannot decode the raw SAFE L0 ISP ourselves (our CCSDS-122 codec rejects the stream; the s2msi
processor and eopf-cpm consume an *already-decompressed* EOPF zarr Synthetic L0). But that decompressed reference raw
DN is published for the **exact same acquisition** (2018-08-20, tile **T36TUL** = turkey) in the s2msi
validation set: `dpr-s2-input/Validation/T36TUL/S02A_MSI_L0_T36TUL.zarr` — read with the **`s2input`** S3
alias. This cell downloads + unzips it (skips if already present). `measurements/d01/b03/img` = reference raw DN.

In [14]:
# 8) fetch EOPF zarr L0 (dpr-s2-input/Validation/T36TUL via the s2input alias)
import json
import os
import shutil
import zipfile
from pathlib import Path

import boto3
from botocore.config import Config

# locate .eopf/secrets.json (env override, ~/.eopf, or the validation path) and its s2input alias
_cands = [os.environ.get("EOPF_SECRETS"),
          os.path.expanduser("~/.eopf/secrets.json")]
SECRETS = next((p for p in _cands if p and os.path.exists(p)), None)
assert SECRETS, "no .eopf/secrets.json found (set EOPF_SECRETS)"
e = json.loads(Path(SECRETS).read_text())["s2input"]
ak = e.get("access_key") or e.get("access_key_id") or e.get("key")
sk = e.get("secret") or e.get("secret_access_key")
ep = (e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/")
rg = e.get("region_name", "sbg")
print(f"secrets: {SECRETS}\ns2input endpoint: {ep}")

s3 = boto3.client("s3", endpoint_url=ep, region_name=rg,
                  aws_access_key_id=ak, aws_secret_access_key=sk,
                  config=Config(s3={"addressing_style": "path"}, connect_timeout=20, retries={"max_attempts": 3}))

BUCKET, PREFIX = "dpr-s2-input", "Validation/T36TUL/"
KEY = PREFIX + "S02A_MSI_L0_T36TUL.zarr.zip"
DEST = Path("/home/jovyan/data-store/dpr-common-validation/T36TUL")
ZARR = DEST / "S02A_MSI_L0_T36TUL.zarr"

# verify access first
try:
    r = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=40)
    keys = [(o["Key"], o["Size"]) for o in r.get("Contents", [])]
    print(f"\naccess OK — {len(keys)} objects under {PREFIX}:")
    for k, sz in keys[:15]:
        print(f"   {sz/2**20:8.1f} MB  {k.split('/')[-1]}")
except Exception as exc:
    code = getattr(exc, "response", {}).get("Error", {}).get("Code", type(exc).__name__)
    raise SystemExit(f"cannot access {BUCKET}/{PREFIX}: {code} — the s2input key may lack access")

if ZARR.exists():
    print(f"\nalready present: {ZARR}")
else:
    DEST.mkdir(parents=True, exist_ok=True)
    probe = DEST
    while not probe.exists():
        probe = probe.parent
    free = shutil.disk_usage(probe).free
    zsize = next((sz for k, sz in keys if k.endswith("S02A_MSI_L0_T36TUL.zarr.zip")), 0)
    print(f"\nfree disk: {free/2**30:.1f} GB | zip: {zsize/2**30:.2f} GB")
    zpath = DEST / "S02A_MSI_L0_T36TUL.zarr.zip"
    print("downloading ...", flush=True)
    s3.download_file(BUCKET, KEY, str(zpath))
    print("unzipping ...", flush=True)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(ZARR)
    zpath.unlink()
    print(f"done -> {ZARR}")


KeyError: 's2input'

## 9) Synthetic L0 vs EOPF-zarr L0 (scene + distribution)

Compare our generated synthetic L0 raw (`L0_synth` from the step-by-step cell) against the **reference raw
DN** just fetched (`measurements/d01/b03/img`). Same acquisition (2018-08-20 T36TUL), but our synthetic
L0 comes from the N05.11-reprocessed L1B re-impressed with a 2024 ESA ADF while the ESA L0 is the
operationally-processed raw — so this is a **scene + distribution** check (same scene, comparable DN
range/character), not a pixel-exact match. Panels: two images + overlaid DN histograms; stats printed.

In [ ]:
# 9) synthetic L0  vs  EOPF-zarr L0  (scene + distribution)
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from s2_msi_raw_generator import forward_radiometric_atbd as fwd, io as gio

BAND, DET, LINES = "B03", 1, 1024
ZARR = Path("/home/jovyan/data-store/dpr-common-validation/T36TUL/S02A_MSI_L0_T36TUL.zarr")


def _resolve_zarr(p):
    p = Path(p)
    if (p / ".zgroup").exists() or (p / "zarr.json").exists() or (p / "measurements").exists():
        return str(p)
    subs = [c for c in p.iterdir() if c.is_dir() and c.name.endswith(".zarr")] if p.exists() else []
    return str(subs[0]) if subs else str(p)


def _st(a):
    v = a[a > 0]; v = v if v.size else a
    return f"mean={v.mean():.1f} std={v.std():.1f} min={v.min():.0f} max={v.max():.0f} FPN={fwd.column_fpn(a):.4f}"


esa_l0 = None
if ZARR.exists():
    try:
        esa_l0 = gio.read_l1b_band(_resolve_zarr(ZARR), DET, BAND, lines=slice(0, LINES))
        esa_l0 = np.rot90(esa_l0, k=1)      # match the synthetic readout orientation (k=3 for clockwise)
        print(f"ESA L0 img: measurements/d{DET:02d}/{BAND.lower()}/img  shape={esa_l0.shape}")
    except Exception as exc:
        print("could not read EOPF zarr Synthetic L0:", type(exc).__name__, exc)
else:
    print("EOPF zarr L0 not found — run the fetch cell (8) first:", ZARR)

if "L0_synth" not in dir():
    print("run the step-by-step cell (7) first — it produces L0_synth")
elif esa_l0 is None:
    print("no ESA L0 to compare (see fetch cell 8)")
else:
    print("synthetic L0 :", _st(np.asarray(L0_synth, float)))
    print("ESA L0 ref  :", _st(esa_l0))
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
    for k, (img, t) in enumerate([(np.asarray(L0_synth, float), "synthetic L0 (ours)"),
                                  (esa_l0, "ESA L0 (T36TUL)")]):
        m = img[img > 0]; lo, hi = np.percentile(m, (2, 98))
        im = ax[k].imshow(np.clip(img, lo, hi), cmap="gray", aspect="auto", interpolation="nearest")
        ax[k].set_title(f"{t}\n{_st(img)}", fontsize=8); ax[k].set_xticks([]); ax[k].set_yticks([])
        plt.colorbar(im, ax=ax[k], fraction=0.046, pad=0.02)
    ax[2].hist(np.asarray(L0_synth, float)[L0_synth > 0].ravel(), bins=80, alpha=0.6, density=True, label="synthetic")
    ax[2].hist(esa_l0[esa_l0 > 0].ravel(), bins=80, alpha=0.6, density=True, label="ESA L0")
    ax[2].set_title("Synthetic L0 DN histogram (scene + distribution)"); ax[2].set_xlabel("DN"); ax[2].legend(fontsize=8)
    fig.suptitle(f"Synthetic L0 vs EOPF-zarr L0 — {BAND} d{DET:02d}, T36TUL 2018-08-20 (same acquisition)",
                 fontsize=10)
    fig.tight_layout(); plt.show()


## 9) Diagnose the ESA L0 tar (what is actually inside?)

The side-by-side looked wrong, so before trusting any "reference ESA L0" image we dump the ESA L0 tar
structure: member names + sizes, grouped by type, and for every jp2 its name + shape. S2 L0 is raw
**ISP** (compressed source packets) — if there is no true per-detector science image, the earlier
extraction grabbed a preview/mask/wrong detector, and an comparison needs the ground-decode step.

In [12]:
# 9) inspect the ESA L0 tar contents (nested)
import io as _io
import os
import tarfile
from collections import Counter

L0_DIR = "/home/jovyan/validation-data/data-store/dpr-common-validation/turkey"
L0_TARS = sorted([os.path.join(L0_DIR, f) for f in os.listdir(L0_DIR)
                  if "MSI_L0" in f and f.endswith(".tar")], key=os.path.getsize)

def _walk(tar, prefix="", depth=0, jp2s=None, ext=None):
    for m in tar.getmembers():
        if not m.isfile():
            continue
        nm = prefix + m.name
        e = os.path.splitext(m.name)[1].lower() or "(none)"
        ext[e] += 1
        if m.name.endswith(".tar") and depth < 2:
            try:
                with tarfile.open(fileobj=_io.BytesIO(tar.extractfile(m).read())) as inn:
                    _walk(inn, nm + "::", depth + 1, jp2s, ext)
            except Exception as exc:
                print("   (nested open failed:", m.name, exc, ")")
        elif m.name.endswith(".jp2"):
            jp2s.append((nm, m.size))

for t in L0_TARS:
    print("=" * 90)
    print(os.path.basename(t), f"({os.path.getsize(t)/2**20:.1f} MB)")
    ext, jp2s = Counter(), []
    with tarfile.open(t) as tar:
        _walk(tar, jp2s=jp2s, ext=ext)
    print("  file types:", dict(ext))
    print(f"  jp2 files: {len(jp2s)}")
    for nm, sz in jp2s[:25]:
        # decode shape (cheap) to tell science image (big) from mask/preview (small)
        tag = ""
        try:
            from PIL import Image
            # re-open to read this jp2 (nested-safe path handling omitted for brevity: show name+size)
        except Exception:
            pass
        print(f"     {sz/1024:8.0f} KB  {nm[-95:]}")
    # per-detector / per-band presence in names
    import re
    dets = sorted(set(re.findall(r"_D(\d{2})_", " ".join(n for n, _ in jp2s))))
    bands = sorted(set(re.findall(r"_(B\d\d|B8A)\.jp2", " ".join(n for n, _ in jp2s))))
    print("  detectors in jp2 names:", dets or "none", "| bands:", bands or "none")


S2A_OPER_MSI_L0__DS_MPS__20180820T103944_S20180820T084811_N02.06.tar (31.4 MB)
  file types: {'.bin': 28, '.safe': 1, '.xml': 6, '.xsd': 3, '.jp2': 5}
  jp2 files: 5
         1229 KB  0180820T084811_N02.06/QI_DATA/S2A_OPER_QLK_L0__DS_MPS__20180820T103944_S20180820T084811_B10.jp2
         3473 KB  0180820T084811_N02.06/QI_DATA/S2A_OPER_QLK_L0__DS_MPS__20180820T103944_S20180820T084811_B11.jp2
         3172 KB  0180820T084811_N02.06/QI_DATA/S2A_OPER_QLK_L0__DS_MPS__20180820T103944_S20180820T084811_B03.jp2
         3230 KB  0180820T084811_N02.06/QI_DATA/S2A_OPER_QLK_L0__DS_MPS__20180820T103944_S20180820T084811_B04.jp2
         2995 KB  0180820T084811_N02.06/QI_DATA/S2A_OPER_QLK_L0__DS_MPS__20180820T103944_S20180820T084811_B02.jp2
  detectors in jp2 names: none | bands: ['B02', 'B03', 'B04', 'B10', 'B11']
aux_S2A_OPER_MSI_L0__DS_MPS__20180820T103944_S20180820T084811_N02.06.tar (397.2 MB)
  file types: {'.tgz': 157}
  jp2 files: 0
  detectors in jp2 names: none | bands: none


## 10) Attempt: decode the ESA L0 ISP -> raw DN

The raw is in the CCSDS-122 ISP `.bin` packets. This lists the `.bin` members, then **attempts**
a decode with the repo's tools (`sad.scan_ccsds_packets` + `ccsds122.decompress_frame`). This is an
honest attempt — S2's operational ISP framing/CCSDS-122 profile may differ from this repo's, so it may
not yield a clean image; the cell reports exactly how far it gets.

In [13]:
# 10) attempt ISP decode
import io as _io
import os
import re
import tarfile

import numpy as np

from s2_msi_raw_generator import ccsds122, sad

L0_DIR = "/home/jovyan/validation-data/data-store/dpr-common-validation/turkey"
L0_TAR = next(os.path.join(L0_DIR, f) for f in os.listdir(L0_DIR)
              if "MSI_L0" in f and f.endswith(".tar") and "aux" not in f)

with tarfile.open(L0_TAR) as tar:
    bins = [(m.name, m.size, m) for m in tar.getmembers() if m.isfile() and m.name.endswith(".bin")]
    print(f"{len(bins)} .bin ISP members (name : KB):")
    for nm, sz, _ in sorted(bins, key=lambda x: -x[1])[:30]:
        print(f"   {sz/1024:8.0f} KB  {nm.split('/')[-1]}")

    # try the largest .bin: scan CCSDS packets, look at structure, attempt a decompress
    if bins:
        nm, sz, m = max(bins, key=lambda x: x[1])
        buf = tar.extractfile(m).read()
        print(f"\nscanning {nm.split('/')[-1]} ({sz/1024:.0f} KB) ...")
        try:
            pkts = sad.scan_ccsds_packets(buf)
            print(f"  CCSDS packets found: {len(pkts)}")
            if pkts:
                from collections import Counter
                apids = Counter(p.get("apid") for p in pkts)
                print("  APIDs:", dict(apids))
                # naive payload join for the dominant APID, then try CCSDS-122 decode
                dom = apids.most_common(1)[0][0]
                payload = b"".join(buf[p["offset"]:p["offset"] + 6 + p["data_len"] + 1][6:]
                                   for p in pkts if p.get("apid") == dom)
                print(f"  dominant APID {dom}: {len([1 for p in pkts if p.get('apid')==dom])} pkts, "
                      f"~{len(payload)/1024:.0f} KB payload")
                try:
                    frame = ccsds122.decompress_frame(payload)
                    print(f"  ✓ decompress_frame OK -> shape {np.shape(frame)}  "
                          f"mean={np.mean(frame):.1f}")
                    print("  (if this looks like a reference image, we can wire a proper per-detector decode)")
                except Exception as exc:
                    print(f"  ✗ decompress_frame failed: {type(exc).__name__}: {exc}")
                    print("  -> the S2 CCSDS-122/ISP profile differs from this repo's codec;")
                    print("     a faithful ESA-L0 decode needs the operational L1A decompressor.")
        except Exception as exc:
            print("  scan_ccsds_packets failed:", type(exc).__name__, exc)


28 .bin ISP members (name : KB):
       1582 KB  S2A_OPER_AUX_S37105_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN.bin
       1582 KB  S2A_OPER_AUX_S38105_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN.bin
       1305 KB  S2A_OPER_AUX_S11121_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN.bin
        294 KB  S2A_OPER_AUX_S11105_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN.bin
        294 KB  S2A_OPER_AUX_S11108_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN.bin
        290 KB  S2A_OPER_AUX_S11118_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN.bin
        290 KB  S2A_OPER_AUX_S11116_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN.bin
        290 KB  S2A_OPER_AUX_S11113_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN.bin
        290 KB  S2A_OPER_AUX_S11115_MPS__20180820T103944_V20180820T083601_20180820T090911_A000064_WF_LN

## Scan: which L1B products are near the ESA ADF epoch (2024-04-17)?

Our ESA R2EQOG ADF is dated **2024-04-17** (both S2A and S2B). For a temporally-valid ESA-only run we
need an **L1B** (sensor geometry) captured near that date. This cell scans the accessible buckets on the
VM — `dpr-common` (via `s3input`) and `dpr-s2-input` (via `s2input`) — for any L1B product (`.SAFE`,
`_L1B.tar` bundle, or `.zarr`), extracts its sensing date, and ranks by proximity to 2024-04-17.

In [ ]:
# scan accessible buckets for L1B products, ranked by |sensing - 2024-04-17|
import json
import os
import re
from datetime import date

import boto3
from botocore.config import Config

_cands = [os.environ.get("EOPF_SECRETS"), os.path.expanduser("~/.eopf/secrets.json")]
SECRETS = next((p for p in _cands if p and os.path.exists(p)), None)
assert SECRETS, "no .eopf/secrets.json (set EOPF_SECRETS)"
raw = json.loads(Path(SECRETS).read_text()) if False else json.loads(open(SECRETS).read())
ADF = date(2024, 4, 17)

def _client(alias):
    e = raw[alias]
    return boto3.client(
        "s3",
        endpoint_url=(e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/"),
        region_name=e.get("region_name", "sbg"),
        aws_access_key_id=e.get("access_key") or e.get("access_key_id") or e.get("key"),
        aws_secret_access_key=e.get("secret") or e.get("secret_access_key"),
        config=Config(s3={"addressing_style": "path"}, connect_timeout=20, retries={"max_attempts": 2}),
    )

def _sens(name):
    for pat in (r"MSIL1B_(\d{8})T", r"_S(\d{8})T", r"_V(\d{8})T", r"(\d{8})T\d{6}"):
        m = re.search(pat, name)
        if m:
            dt = m.group(1)
            try:
                return date(int(dt[:4]), int(dt[4:6]), int(dt[6:8]))
            except ValueError:
                pass
    return None

TARGETS = [
    ("s3input", "dpr-common", ["S2AMSIdataset/", "S2BMSIdataset/"]),
    ("s2input", "dpr-s2-input", ["Validation/"]),
]
IS_L1B = re.compile(r"_L1B\.tar$|MSIL1B[^/]*\.(SAFE|zarr)(\.zip)?$|MSI_L1B__DS|[^/]*L1B[^/]*\.zarr$")

hits = {}   # (bucket, dir) -> (date, satellite)
for alias, bucket, prefixes in TARGETS:
    try:
        s3 = _client(alias)
    except KeyError:
        print(f"[{alias}] not in secrets — skip"); continue
    for pfx in prefixes:
        try:
            tok, n = None, 0
            while True:
                kw = dict(Bucket=bucket, Prefix=pfx, MaxKeys=1000)
                if tok:
                    kw["ContinuationToken"] = tok
                r = s3.list_objects_v2(**kw)
                for o in r.get("Contents", []):
                    k = o["Key"]; n += 1
                    if "L1B" not in k and "l1b" not in k:
                        continue
                    base = k.split("/")[-1]
                    if not (IS_L1B.search(k) or k.rstrip("/").endswith("_L1B") or "MSIL1B" in base):
                        continue
                    d = _sens(base) or _sens(k)
                    sat = "S2A" if re.search(r"S2A|S02A", k) else ("S2B" if re.search(r"S2B|S02B", k) else "?")
                    root = "/".join(k.split("/")[:-1]) or k
                    prev = hits.get((bucket, root))
                    if prev is None or (d and (prev[0] is None or d < prev[0])):
                        hits[(bucket, root)] = (d, sat)
                if r.get("IsTruncated"):
                    tok = r.get("NextContinuationToken")
                else:
                    break
            print(f"[{alias}] {bucket}/{pfx}: scanned {n} keys")
        except Exception as exc:
            code = getattr(exc, "response", {}).get("Error", {}).get("Code", type(exc).__name__)
            print(f"[{alias}] {bucket}/{pfx}: {code}")

print("\n=== L1B products found (sensing | gap to ADF | sat | location) ===")
rows = sorted(((abs((d - ADF).days) if d else 10**9, d, sat, f"{b}/{root}")
               for (b, root), (d, sat) in hits.items()))
for gap, d, sat, loc in rows:
    tag = "  <<< 2024 NEAR" if gap < 300 else ""
    g = f"{gap}d" if gap < 10**9 else "?"
    print(f"  {str(d):12} gap={g:>7} {sat:4}{tag}  {loc[:80]}")
if not rows:
    print("  (no L1B products found in the accessible buckets)")
